## Méthode hybride: CNN + RB 


In [ ]:
import pandas as pd
import numpy as np
import joblib
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination


# =========================
# 1) Charger les objets sauvegardés
# =========================
print("Chargement des données et du modèle... - cnn+BN.ipynb:20")


# Prédictions sauvegardées
train_preds_bin = joblib.load("train_preds_bin.pkl")
test_preds_bin = joblib.load("test_preds_bin.pkl")

# Données train/test
X_train = joblib.load("X_train.pkl")
X_test = joblib.load("X_test.pkl")
y_train = joblib.load("y_train.pkl")
y_test = joblib.load("y_test.pkl")

# features
features = joblib.load("features.pkl")

In [ ]:
# =========================
# 2) Fonction de discrétisation
# =========================
def discretize_train_test(train_col, test_col, n_bins=5):
    # Création des bornes à partir du train
    # edges: les bornes de chaque tranche
    _, edges = pd.qcut(train_col, q=n_bins, duplicates="drop", retbins=True)

    # Discrétisation train
    # transforme les valeurs numérique en catégorie selon les bornes
    train_disc = pd.cut(
        train_col, bins=edges, labels=False, include_lowest=True
    ).astype(float)

    # Discrétisation test avec les mêmes bornes
    test_disc = pd.cut(test_col, bins=edges, labels=False, include_lowest=True).astype(
        float
    )

    # Gestion des NaN -> mettre 0
    return train_disc.fillna(0).astype(int), test_disc.fillna(0).astype(int)

In [ ]:
# =========================
# 3) Préparer les datasets pour le RB
# =========================
print("Préparation des données pour le Réseau Bayésien... - cnn+BN.ipynb:4")

df_train_bn = pd.DataFrame()
df_test_bn = pd.DataFrame()

for idx, col in enumerate(features):
    train_disc, test_disc = discretize_train_test(
        pd.Series(X_train[:, idx]), pd.Series(X_test[:, idx])
    )
    df_train_bn[col + "_bin"] = train_disc
    df_test_bn[col + "_bin"] = test_disc
    # toutes les colonnes continues sont transformées en colonnes discrètes pour le RB.
# Ajout des prédictions CNN
df_train_bn["cnn_pred"] = train_preds_bin
df_test_bn["cnn_pred"] = test_preds_bin

# Ajout de la variable cible
df_train_bn["went_on_backorder"] = y_train.values

In [ ]:
# =========================
# 4) Construire le Réseau Bayésien
# =========================
print("Construction du Réseau Bayésien... - cnn+BN.ipynb:4")

edges = [(col + "_bin", "went_on_backorder") for col in features] + [
    ("cnn_pred", "went_on_backorder")
]

model_bn = DiscreteBayesianNetwork(edges)
model_bn.fit(df_train_bn, estimator=MaximumLikelihoodEstimator)
infer_bn = VariableElimination(model_bn)

In [ ]:
# =========================
# 5) Inférence sur le test
# =========================
print("Inférence et évaluation... - cnn+BN.ipynb:8")

hybrid_preds = []
for i in range(len(df_test_bn)):
    evidence = {col + "_bin": int(df_test_bn.iloc[i][col + "_bin"]) for col in features}
    evidence["cnn_pred"] = int(df_test_bn.iloc[i]["cnn_pred"])

    q = infer_bn.query(["went_on_backorder"], evidence=evidence, show_progress=False)
    hybrid_preds.append(np.argmax(q.values))

In [ ]:

# =========================
# 6) Évaluation finale
# =========================
print("Résultats finaux : - cnn+BN.ipynb:5")
print("Accuracy : - cnn+BN.ipynb:6", accuracy_score(y_test, hybrid_preds))
print("Precision : - cnn+BN.ipynb:7", precision_score(y_test, hybrid_preds))
print("Recall : - cnn+BN.ipynb:8", recall_score(y_test, hybrid_preds))
print("F1 : - cnn+BN.ipynb:9", f1_score(y_test, hybrid_preds))
print("ROC AUC : - cnn+BN.ipynb:10", roc_auc_score(y_test, hybrid_preds))

Chargement des données et du modèle... - cnn+BN.ipynb:20
Préparation des données pour le Réseau Bayésien... - cnn+BN.ipynb:63
Construction du Réseau Bayésien... - cnn+BN.ipynb:86


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'national_inv_bin': 'N', 'lead_time_bin': 'N', 'in_transit_qty_bin': 'N', 'forecast_3_month_bin': 'N', 'forecast_6_month_bin': 'N', 'forecast_9_month_bin': 'N', 'sales_1_month_bin': 'N', 'sales_3_month_bin': 'N', 'sales_6_month_bin': 'N', 'sales_9_month_bin': 'N', 'min_bank_bin': 'N', 'potential_issue_bin': 'N', 'pieces_past_due_bin': 'N', 'perf_6_month_avg_bin': 'N', 'perf_12_month_avg_bin': 'N', 'local_bo_qty_bin': 'N', 'deck_risk_bin': 'N', 'oe_constraint_bin': 'N', 'ppap_risk_bin': 'N', 'stop_auto_buy_bin': 'N', 'rev_stop_bin': 'N', 'cnn_pred': 'N', 'went_on_backorder': 'N'}


Inférence et évaluation... - cnn+BN.ipynb:100
Résultats finaux : - cnn+BN.ipynb:114
Accuracy : - cnn+BN.ipynb:115 0.9263600684731327
Precision : - cnn+BN.ipynb:116 0.911437286319555
Recall : - cnn+BN.ipynb:117 0.9443801144808075
F1 : - cnn+BN.ipynb:118 0.9276163144340852
ROC AUC : - cnn+BN.ipynb:119 0.9263727423841975


In [3]:
import joblib
joblib.dump(model_bn, "bayesian_model.pkl")

['bayesian_model.pkl']

In [4]:
discretization_edges = {}
for idx, col in enumerate(features):
    _, edges = pd.qcut(X_train[:, idx], q=5, duplicates="drop", retbins=True)
    discretization_edges[col] = edges
joblib.dump(discretization_edges, "discretization_edges.pkl")

['discretization_edges.pkl']

In [2]:
import pandas as pd
import numpy as np
import joblib
from tensorflow.keras.models import load_model
from pgmpy.inference import VariableElimination

# ============================
# 1) Charger les modèles et edges
# ============================
cnn_model = load_model("cnn_model.h5")  # ton CNN
model_bn = joblib.load("bayesian_model.pkl")  # BN
discretization_edges = joblib.load(
    "discretization_edges.pkl"
)  # edges de discrétisation
scaler = joblib.load("scaler.pkl")  # scaler utilisé pendant training

# ============================
# 2) Charger dataset de test
# ============================
df_test = pd.read_csv("Datasets/testing_BOP.csv")

# Supprimer la colonne cible si elle existe
if "went_on_backorder" in df_test.columns:
    df_test = df_test.drop(columns=["went_on_backorder"])

# ============================
# 3) Nettoyage et encodage
# ============================
cat_cols = [
    "potential_issue",
    "deck_risk",
    "oe_constraint",
    "ppap_risk",
    "stop_auto_buy",
    "rev_stop",
]

df_test = df_test.replace(-99, np.nan)

for col in df_test.columns:
    if col in cat_cols:
        df_test[col] = df_test[col].fillna(df_test[col].mode()[0])
        df_test[col] = df_test[col].map({"No": 0, "Yes": 1})
    else:
        if pd.api.types.is_numeric_dtype(df_test[col]):
            df_test[col] = df_test[col].fillna(df_test[col].median())
        else:
            df_test[col] = df_test[col].fillna(df_test[col].mode()[0])

# ============================
# 4) Standardisation pour CNN
# ============================
X_test = df_test.drop(columns=["sku"], errors="ignore")
X_test_scaled = scaler.transform(X_test)

# Reshape pour CNN 1D
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

# ============================
# 5) Prédiction CNN
# ============================
cnn_preds_prob = cnn_model.predict(X_test_cnn).ravel()
cnn_preds_bin = (cnn_preds_prob > 0.5).astype(int)

# ============================
# 6) Préparation pour BN
# ============================
df_test_bn = pd.DataFrame()
for col, edges in discretization_edges.items():
    if col in df_test.columns:
        df_test_bn[col + "_bin"] = (
            pd.cut(df_test[col], bins=edges, labels=False, include_lowest=True)
            .fillna(0)
            .astype(int)
        )

# Ajouter prédiction CNN comme feature
df_test_bn["cnn_pred"] = cnn_preds_bin

# ============================
# 7) Inférence BN
# ============================
infer = VariableElimination(model_bn)
bn_preds = []
bn_probs = []

for i in range(len(df_test_bn)):
    evidence = {col: int(df_test_bn.iloc[i][col]) for col in df_test_bn.columns}
    q = infer.query(
        variables=["went_on_backorder"], evidence=evidence, show_progress=False
    )
    prob = q.values[1]  # probabilité que went_on_backorder=1
    bn_probs.append(prob)
    bn_preds.append(1 if prob > 0.5 else 0)

# ============================
# 8) Sauvegarder les résultats
# ============================
df_results = df_test.copy()
df_results["pred_class"] = bn_preds
df_results["pred_prob"] = bn_probs

df_results.to_csv("hybrid_CNN_BN_predictions.csv", index=False)
print(
    "✅ Prédictions hybrides CNN + BN sauvegardées dans 'hybrid_CNN_BN_predictions.csv'"
)

C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update th

7565/7565 ━━━━━━━━━━━━━━━━━━━━ 11s 1ms/step
✅ Prédictions hybrides CNN + BN sauvegardées dans 'hybrid_CNN_BN_predictions.csv'


In [6]:
# Charger les résultats si besoin
df_results = pd.read_csv("hybrid_CNN_BN_predictions.csv")

# Afficher les 10 premières lignes avec les colonnes prédiction
df_results[["pred_class", "pred_prob"]].head(10)

C:\Users\hp\AppData\Local\Temp\ipykernel_15216\1877763591.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_results = pd.read_csv("hybrid_CNN_BN_predictions.csv")


,pred_class,pred_prob
0,0,0.032698
1,0,0.032698
2,0,0.074400
3,0,0.085020
4,0,0.052013
5,0,0.000000
6,0,0.032698
7,0,0.032698
8,0,0.032698
9,0,0.032698
